# 03 — Feature Engineering, Leakage Audit & Train/Test Split

## Objective
Build the leakage-safe modeling dataset.

This notebook:
- separates predictors from outcomes/treatment
- encodes categorical customer attributes
- preserves treatment assignment separately
- performs a reproducible stratified train/test split
- fits the preprocessing pipeline on training data only
- saves preprocessing artifacts

In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

# Find the project root so the notebook works whether launched from the project root
# or from inside the notebooks directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
ROOT = next((p for p in _candidates if (p / "src" / "config.py").exists()), _here)
if not (ROOT / "src" / "config.py").exists():
    raise FileNotFoundError("Could not locate project root containing src/config.py")

sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from src.data_utils import load_data, prepare
from src.feature_engineering import make_preprocessor
from src.config import DATA_PATH, RANDOM_SEED, TEST_SIZE, FEATURES, MODELS_DIR

df = prepare(load_data(DATA_PATH))
print("Rows:", len(df))
print("Features:", FEATURES)

## 1. Leakage audit

In [ ]:
outcome_columns = {"visit", "conversion", "spend"}
forbidden = outcome_columns.intersection(FEATURES)
print("Outcome columns accidentally included as predictors:", forbidden)
assert not forbidden

print("Treatment column included in predictors:", "segment" in FEATURES)
assert "segment" not in FEATURES

## 2. Reproducible stratified split

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df["segment"]
)

train = df.iloc[idx_train].reset_index(drop=True)
test = df.iloc[idx_test].reset_index(drop=True)

print("Train:", train.shape, "Test:", test.shape)
print("\nTrain treatment distribution:")
display(train["segment"].value_counts(normalize=True).rename("share").to_frame())
print("\nTest treatment distribution:")
display(test["segment"].value_counts(normalize=True).rename("share").to_frame())

## 3. Fit preprocessing only on the training set

In [ ]:
pre = make_preprocessor()
X_train = pre.fit_transform(train)
X_test = pre.transform(test)
X_all = pre.transform(df)

print("Encoded train matrix:", X_train.shape)
print("Encoded test matrix:", X_test.shape)

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pre, MODELS_DIR / "notebook_03_preprocessor.joblib")
joblib.dump(FEATURES, MODELS_DIR / "notebook_03_feature_columns.joblib")

pd.DataFrame({
    "split":["train","test","all"],
    "rows":[len(train),len(test),len(df)],
}).to_csv(ROOT / "artifacts" / "metrics" / "notebook_03_split_summary.csv", index=False)

print("Saved preprocessing artifacts.")

## 4. Feature matrix sanity checks

In [ ]:
assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()
assert X_train.shape[1] == X_test.shape[1]
print("No non-finite encoded values detected.")

## Conclusion
The modeling data is now prepared without leaking visit, conversion, spend, or treatment assignment into customer predictors.

The next notebook trains the actual uplift models on the training split.